## Embeddings

Generate vector embeddings for text chunks using HuggingFace sentence-transformers.

> **About sentence-transformers:** A Python library for computing dense vector representations of sentences. It uses pre-trained models like BERT to generate embeddings that capture semantic meaning, enabling similarity search and clustering.

### Import Libraries

In [1]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Global model constant - load once, reuse everywhere
MODEL_NAME = "all-MiniLM-L6-v2"
model = SentenceTransformer(MODEL_NAME)

### What are Embeddings?

> **Note:** Embeddings convert text into numerical vectors where:
> - Similar meanings = vectors close together
> - Different meanings = vectors far apart
> - Enables semantic search (not just keyword matching)

### Generate Embeddings for Sample Texts

In [2]:
# Sample legal phrases for embedding
texts = [
    "Contract termination clause",
    "Agreement ending provision",
    "Forest conservation laws",
    "Environmental protection regulations",
    "The tiger perishes without the forest"
]

# Generate embeddings using global model
embeddings = model.encode(texts)

print(f"Generated embeddings shape: {embeddings.shape}")
print(f"  {len(texts)} texts → {embeddings.shape[1]} dimensional vectors")

Generated embeddings shape: (5, 384)
  5 texts → 384 dimensional vectors


### Explore Embedding Vector

In [3]:
# Print first embedding (first 20 dimensions)
print(f"First embedding (showing 20 of {len(embeddings[0])} dimensions):")
print(embeddings[0][:20])

First embedding (showing 20 of 384 dimensions):
[-0.03245153  0.121222    0.0766131  -0.02901453 -0.01007965  0.07525725
  0.04213667 -0.01699385  0.06639607  0.03980721  0.02807431  0.04919358
  0.01791115 -0.00647959  0.00866416  0.0463797  -0.05176838  0.04586427
 -0.02105638  0.00774302]


### Calculate Semantic Similarity

In [4]:
# Calculate cosine similarity between all pairs
similarity_matrix = cosine_similarity(embeddings)

print("Similarity matrix (5x5):")
print(np.round(similarity_matrix, 2))

Similarity matrix (5x5):
[[1.   0.63 0.24 0.12 0.09]
 [0.63 1.   0.21 0.18 0.06]
 [0.24 0.21 1.   0.46 0.39]
 [0.12 0.18 0.46 1.   0.07]
 [0.09 0.06 0.39 0.07 1.  ]]


### Find Most Similar Pairs

In [5]:
# Print similarity between specific pairs
pairs = [
    (0, 1, "Contract termination vs Agreement ending"),
    (2, 3, "Forest conservation vs Environmental protection"),
    (0, 2, "Contract vs Forest (unrelated)"),
    (4, 2, "Tiger/forest quote vs Forest laws")
]

print("Semantic similarity between pairs:")
for i, j, label in pairs:
    sim = similarity_matrix[i][j]
    print(f"  {label}: {sim:.3f}")

Semantic similarity between pairs:
  Contract termination vs Agreement ending: 0.633
  Forest conservation vs Environmental protection: 0.457
  Contract vs Forest (unrelated): 0.237
  Tiger/forest quote vs Forest laws: 0.395


### Embed Extracted Document Chunks

In [6]:
# Load chunks from previous notebooks
from llama_index.core import SimpleDirectoryReader
from llama_index.core.node_parser import SentenceSplitter
from pathlib import Path

# Load extracted txt files
TXT_DIR = Path("../data/txt")
documents = SimpleDirectoryReader(str(TXT_DIR)).load_data()

# Split into chunks
splitter = SentenceSplitter(chunk_size=512, chunk_overlap=50)
nodes = splitter.get_nodes_from_documents(documents)

print(f"Loaded {len(nodes)} chunks for embedding")

Loaded 57 chunks for embedding


### Generate Embeddings for All Chunks

In [7]:
# Extract text from all chunks
chunk_texts = [node.text for node in nodes]

# Generate embeddings (this may take a minute)
chunk_embeddings = model.encode(chunk_texts)


print(f"Generated embeddings for {len(chunk_embeddings)} chunks")
print(f"Embedding shape: {chunk_embeddings.shape}")

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Generated embeddings for 57 chunks
Embedding shape: (57, 384)


### Find Similar Chunks to a Query

In [8]:
# Encode a query
query = "What are the forest conservation issues?"
query_embedding = model.encode([query])

# Calculate similarity to all chunks
similarities = cosine_similarity(query_embedding, chunk_embeddings)[0]

# Get top 3 most similar chunks
top_indices = np.argsort(similarities)[::-1][:3]

print(f"Query: {query}\n")
print("Top 3 most similar chunks:")
for i, idx in enumerate(top_indices, 1):
    print(f"\n{i}. Similarity: {similarities[idx]:.3f}")
    print(f"   Source: {nodes[idx].metadata.get('file_name', 'unknown')}")
    print(f"   Text: {nodes[idx].text[:200]}...")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Query: What are the forest conservation issues?

Top 3 most similar chunks:

1. Similarity: 0.579
   Source: A_John_Kennedy_vs_The_State_Of_Tamil_Nadu_on_24_March_2025_1_extracted_from_docx.txt
   Text: “29. Forests not only provide for and facilitate the sustenance of life, but they also continue to protect and foster it. They continue to tackle the ever-increasing carbon dioxide emissions produced ...

2. Similarity: 0.559
   Source: A_John_Kennedy_vs_The_State_Of_Tamil_Nadu_on_24_March_2025_1_extracted_from_docx.txt
   Text: Needless to say, that the forests form the lungs of the ecosystem, and any depletion/destruction of forest areas has a direct impact on the entire environment. The world at large is facing the calamit...

3. Similarity: 0.533
   Source: A_John_Kennedy_vs_The_State_Of_Tamil_Nadu_on_24_March_2025_1_extracted_from_docx.txt
   Text: There is a crying need for a change in our approach. Man being an enlightened species, is expected to act as a trustee of the Earth. It

### Embedding Statistics

In [9]:
# Print embedding statistics
print("=== Embedding Statistics ===")
print(f"\nModel: {MODEL_NAME}")
print(f"Embedding dimensions: {chunk_embeddings.shape[1]}")
print(f"Total chunks embedded: {len(chunk_embeddings)}")
print(f"\nSample similarities:")
print(f"  Mean: {np.mean(similarities):.3f}")
print(f"  Max: {np.max(similarities):.3f}")
print(f"  Min: {np.min(similarities):.3f}")
print(f"  Std: {np.std(similarities):.3f}")

=== Embedding Statistics ===

Model: all-MiniLM-L6-v2
Embedding dimensions: 384
Total chunks embedded: 57

Sample similarities:
  Mean: 0.228
  Max: 0.579
  Min: 0.033
  Std: 0.179
